# E016 — simulateur différentiable de scannabilité, calibré sur les vrais décodeurs

But : remplacer une loss QR dessinée à la main par un petit réseau qui approxime la probabilité de
lecture d'OpenCV, ZBar et ZXing-cpp après dégradations. Le réseau n'est jamais promu parce que sa
loss baisse : il doit améliorer les **vrais décodeurs** sur un jeu holdout et, plus tard, sur des
captures téléphone/impression.

Le notebook :

1. indexe des images générées et des captures physiques optionnelles ;
2. applique les treize scénarios du validateur et demande les labels aux vrais décodeurs ;
3. divise par groupe source/prompt pour éviter une fuite d'images quasi identiques ;
4. entraîne un CNN multi-sorties entièrement différentiable ;
5. mesure AUCPR, ROC-AUC, Brier et calibration par décodeur ;
6. tente une optimisation de pixels bornée et vérifie le résultat avec les vrais décodeurs ;
7. exporte TorchScript seulement si les seuils minimaux sont atteints.


## Garde-fous

```text
image source ─► dégradations réelles ─► labels OpenCV/ZBar/ZXing ─► CNN différentiable
      │                                      │                         │
      └──────── groupe anti-fuite ───────────┘                         ▼
                                                            gradient sur holdout
                                                                     │
                                  vrais décodeurs avant/après ◄───────┘
```

- Pas assez de positifs/négatifs : arrêt, dataset conservé.
- Pas de captures physiques : modèle marqué `digital_only`, jamais production.
- Le CNN peut être trompé adversarialement : seul le gain des vrais décodeurs compte.


In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset

from prooftag_qr.validation import DEFAULT_SCENARIOS, QRValidator

assert torch.cuda.is_available(), 'Le CNN peut tourner sur CPU, mais le pod GPU est recommandé.'
DEVICE = torch.device('cuda')
print(torch.cuda.get_device_name(0))


## 1. Sources, seuils d'arrêt et captures physiques

In [ ]:
EXPERIMENT_NAME = 'e016-differentiable-real-decoder-surrogate-v1'
DEFAULT_PAYLOAD = 'https://ptag.io/t/e014'
SOURCE_RUNS = []  # ex. [Path('/data/notebook-runs/...e014a...'), Path('/data/notebook-runs/...e014b...')]
PHYSICAL_CSV = Path('/data/physical-captures/labels.csv')
IMAGE_SIZE = 256
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 2e-4
MIN_SOURCE_GROUPS = 12
MIN_SAMPLES = 120
MIN_CLASS_COUNT_PER_DECODER = 10
REQUIRE_PHYSICAL_FOR_PRODUCTION = True

RUN_DIR = Path('/data/notebook-runs') / (
    datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + EXPERIMENT_NAME
)
RUN_DIR.mkdir(parents=True)
DATASET_IMAGE_DIR = RUN_DIR / 'labelled-images'
DATASET_IMAGE_DIR.mkdir()

if not SOURCE_RUNS:
    patterns = ['*-e014a-real-qart-exact-adaptive-v1', '*-e014b-freeqr-latent-channel-timestep-v1']
    for pattern in patterns:
        matches = sorted(Path('/data/notebook-runs').glob(pattern))
        if matches:
            SOURCE_RUNS.append(matches[-1])
SOURCE_RUNS = [Path(path) for path in SOURCE_RUNS]
if not SOURCE_RUNS:
    raise FileNotFoundError('Aucune source E014A/E014B. Renseigner SOURCE_RUNS.')

template = pd.DataFrame(columns=[
    'image_path', 'expected_payload', 'group_id', 'device',
    'screen_or_print', 'distance_cm', 'lighting', 'notes',
])
template.to_csv(RUN_DIR / 'physical-captures-template.csv', index=False)
print('Sources :', SOURCE_RUNS)
print('Gabarit physique :', RUN_DIR / 'physical-captures-template.csv')


## 2. Construire le dataset avec les vrais décodeurs

In [ ]:
validator = QRValidator()
decoder_names = [decoder.name for decoder in validator.decoders]
print('Décodeurs :', decoder_names)
if len(decoder_names) < 3:
    raise RuntimeError('E016 exige OpenCV + ZBar + ZXing-cpp pour ne pas suradapter un seul moteur.')


def source_pngs(run_dir):
    for path in run_dir.rglob('*.png'):
        if path.name == 'final.png' and 'frames' not in path.parts:
            yield path


def payload_for_run(run_dir):
    manifest = run_dir / 'manifest.json'
    if manifest.exists():
        return json.loads(manifest.read_text(encoding='utf-8')).get('payload', DEFAULT_PAYLOAD)
    return DEFAULT_PAYLOAD


def context_group_for(run_dir, source_path, manifest):
    prompt_id = manifest.get('prompt_id')
    seed = manifest.get('seed')
    prompt_specs = manifest.get('prompts', [])
    if not prompt_id:
        relative_parts = source_path.relative_to(run_dir).parts
        known_ids = {str(item.get('id')) for item in prompt_specs}
        prompt_id = next((part for part in relative_parts if part in known_ids), None)
    if prompt_id and prompt_specs:
        matching = next(
            (item for item in prompt_specs if str(item.get('id')) == str(prompt_id)),
            {},
        )
        seed = matching.get('seed', seed)
    if not prompt_id:
        raise ValueError(
            f'Contexte prompt/seed introuvable pour {source_path}; '
            'ajouter prompt_id/seed au manifest au lieu de créer une fuite.'
        )
    return hashlib.sha256(
        f'{payload_for_run(run_dir)}:{prompt_id}:{seed}'.encode('utf-8')
    ).hexdigest()[:16]


records = []
for run_dir in SOURCE_RUNS:
    expected_payload = payload_for_run(run_dir)
    manifest_path = run_dir / 'manifest.json'
    manifest = (
        json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest_path.exists() else {}
    )
    for source_path in source_pngs(run_dir):
        source = Image.open(source_path).convert('RGB')
        source_group = context_group_for(run_dir, source_path, manifest)
        for scenario in DEFAULT_SCENARIOS:
            transformed = scenario.apply(source)
            key = hashlib.sha256(
                f'{source_group}:{scenario.name}'.encode('utf-8')
            ).hexdigest()[:20]
            saved_path = DATASET_IMAGE_DIR / f'{key}.jpg'
            transformed.save(saved_path, quality=95)
            row = {
                'image_path': str(saved_path), 'source_path': str(source_path),
                'source_run': run_dir.name, 'source_group': source_group,
                'scenario': scenario.name, 'physical': False,
                'expected_payload': expected_payload,
                'expected_payload_hash': hashlib.sha256(expected_payload.encode()).hexdigest(),
            }
            for decoder in validator.decoders:
                decoded = decoder.decode(transformed)
                row[f'label_{decoder.name}'] = int(decoded == expected_payload)
                row[f'detected_{decoder.name}'] = int(bool(decoded))
            records.append(row)

physical_count = 0
if PHYSICAL_CSV.exists():
    physical_frame = pd.read_csv(PHYSICAL_CSV)
    for index, source_row in physical_frame.iterrows():
        path = Path(source_row.image_path)
        if not path.exists():
            print('Capture absente, ignorée :', path)
            continue
        image = Image.open(path).convert('RGB')
        expected_payload = str(source_row.expected_payload)
        group = str(source_row.group_id)
        saved_path = DATASET_IMAGE_DIR / f'physical-{index:05d}.jpg'
        image.save(saved_path, quality=98)
        row = {
            'image_path': str(saved_path), 'source_path': str(path),
            'source_run': 'physical', 'source_group': f'physical:{group}',
            'scenario': 'physical_original', 'physical': True,
            'expected_payload': expected_payload,
            'expected_payload_hash': hashlib.sha256(expected_payload.encode()).hexdigest(),
        }
        for decoder in validator.decoders:
            decoded = decoder.decode(image)
            row[f'label_{decoder.name}'] = int(decoded == expected_payload)
            row[f'detected_{decoder.name}'] = int(bool(decoded))
        records.append(row)
        physical_count += 1

dataset_frame = pd.DataFrame(records)
dataset_frame.to_csv(RUN_DIR / 'decoder-dataset.csv', index=False)
print('Lignes :', len(dataset_frame), 'groupes :', dataset_frame.source_group.nunique())
print('Captures physiques :', physical_count)
display(dataset_frame[[f'label_{name}' for name in decoder_names]].sum().to_frame('positifs'))


## 3. Vérifier l'identifiabilité avant d'entraîner

In [ ]:
problems = []
if len(dataset_frame) < MIN_SAMPLES:
    problems.append(f'{len(dataset_frame)} lignes < {MIN_SAMPLES}')
if dataset_frame.source_group.nunique() < MIN_SOURCE_GROUPS:
    problems.append(
        f'{dataset_frame.source_group.nunique()} groupes < {MIN_SOURCE_GROUPS}'
    )
for name in decoder_names:
    positives = int(dataset_frame[f'label_{name}'].sum())
    negatives = len(dataset_frame) - positives
    if min(positives, negatives) < MIN_CLASS_COUNT_PER_DECODER:
        problems.append(
            f'{name}: classe minoritaire {min(positives, negatives)} < '
            f'{MIN_CLASS_COUNT_PER_DECODER}'
        )
if problems:
    (RUN_DIR / 'STOP-INSUFFICIENT-DATA.json').write_text(
        json.dumps({'problems': problems}, indent=2), encoding='utf-8'
    )
    raise RuntimeError('Dataset non identifiable : ' + '; '.join(problems))


## 4. Split par source, Dataset PyTorch et CNN multi-décodeur

In [ ]:
groups = dataset_frame.source_group.to_numpy()
outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=20260723)
train_val_index, test_index = next(outer.split(dataset_frame, groups=groups))
train_val = dataset_frame.iloc[train_val_index].reset_index(drop=True)
test_frame = dataset_frame.iloc[test_index].reset_index(drop=True)
inner = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=20260724)
train_index, val_index = next(
    inner.split(train_val, groups=train_val.source_group.to_numpy())
)
train_frame = train_val.iloc[train_index].reset_index(drop=True)
val_frame = train_val.iloc[val_index].reset_index(drop=True)
assert set(train_frame.source_group).isdisjoint(val_frame.source_group)
assert set(train_frame.source_group).isdisjoint(test_frame.source_group)
assert set(val_frame.source_group).isdisjoint(test_frame.source_group)
print('train/val/test :', len(train_frame), len(val_frame), len(test_frame))


class DecoderDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(row.image_path).convert('RGB').resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS
        )
        tensor = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0)
        tensor = tensor.permute(2, 0, 1)
        labels = torch.tensor(
            [row[f'label_{name}'] for name in decoder_names], dtype=torch.float32
        )
        return tensor, labels


class ScanSurrogate(nn.Module):
    def __init__(self, outputs):
        super().__init__()
        channels = [3, 32, 64, 128, 192]
        blocks = []
        for source, target in zip(channels[:-1], channels[1:]):
            blocks.extend([
                nn.Conv2d(source, target, 3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(target), nn.SiLU(),
                nn.Conv2d(target, target, 3, padding=1, groups=target, bias=False),
                nn.BatchNorm2d(target), nn.SiLU(),
            ])
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.15),
            nn.Linear(channels[-1], outputs),
        )

    def forward(self, image):
        return self.head(self.features(image))


loaders = {
    'train': DataLoader(DecoderDataset(train_frame), batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
    'val': DataLoader(DecoderDataset(val_frame), batch_size=BATCH_SIZE, shuffle=False, num_workers=2),
    'test': DataLoader(DecoderDataset(test_frame), batch_size=BATCH_SIZE, shuffle=False, num_workers=2),
}
model = ScanSurrogate(len(decoder_names)).to(DEVICE)
positives = torch.tensor(
    [train_frame[f'label_{name}'].sum() for name in decoder_names], dtype=torch.float32
)
negatives = len(train_frame) - positives
criterion = nn.BCEWithLogitsLoss(pos_weight=(negatives / positives.clamp_min(1)).to(DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)


## 5. Entraînement avec sélection sur loss validation

In [ ]:
history = []
best_state = None
best_val = float('inf')
for epoch in range(1, EPOCHS + 1):
    epoch_values = {}
    for phase in ['train', 'val']:
        model.train(phase == 'train')
        total_loss = 0.0
        total_items = 0
        for images, labels in loaders[phase]:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.set_grad_enabled(phase == 'train'):
                logits = model(images)
                loss = criterion(logits, labels)
                if phase == 'train':
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                    optimizer.step()
            total_loss += float(loss.detach()) * len(images)
            total_items += len(images)
        epoch_values[phase] = total_loss / total_items
    history.append({'epoch': epoch, **epoch_values})
    if epoch_values['val'] < best_val:
        best_val = epoch_values['val']
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    print(f"epoch {epoch:02d} train={epoch_values['train']:.4f} val={epoch_values['val']:.4f}")

model.load_state_dict(best_state)
pd.DataFrame(history).to_csv(RUN_DIR / 'training-history.csv', index=False)


## 6. Calibration et holdout sans fuite

In [ ]:
def predict(loader):
    model.eval()
    truths, probabilities = [], []
    with torch.no_grad():
        for images, labels in loader:
            probabilities.append(torch.sigmoid(model(images.to(DEVICE))).cpu().numpy())
            truths.append(labels.numpy())
    return np.concatenate(truths), np.concatenate(probabilities)


y_test, p_test = predict(loaders['test'])
metrics = {}
figure, axes = plt.subplots(1, len(decoder_names), figsize=(6 * len(decoder_names), 5))
axes = np.atleast_1d(axes)
for index, name in enumerate(decoder_names):
    truth, probability = y_test[:, index], p_test[:, index]
    both_classes = len(np.unique(truth)) == 2
    metrics[name] = {
        'average_precision': float(average_precision_score(truth, probability)),
        'brier': float(brier_score_loss(truth, probability)),
        'roc_auc': float(roc_auc_score(truth, probability)) if both_classes else None,
        'positives': int(truth.sum()), 'negatives': int(len(truth) - truth.sum()),
        'holdout_has_both_classes': both_classes,
    }
    observed, predicted = calibration_curve(truth, probability, n_bins=8, strategy='quantile')
    axes[index].plot(predicted, observed, marker='o', label=name)
    axes[index].plot([0, 1], [0, 1], '--', color='gray')
    axes[index].set(xlabel='probabilité prédite', ylabel='fréquence réelle', title=name)
    axes[index].grid(alpha=0.25)
figure.tight_layout()
figure.savefig(RUN_DIR / 'calibration.png', dpi=160)
display(figure)
(RUN_DIR / 'test-metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
display(pd.DataFrame(metrics).T)


## 7. Audit adversarial : le gradient améliore-t-il les vrais décodeurs ?

On choisit un négatif holdout, optimise au plus ±8/255 par pixel avec pénalité TV, puis redemande
les labels aux vrais décodeurs. Une probabilité CNN plus haute sans amélioration réelle est un
échec du surrogate, pas un succès.


In [ ]:
strict_probability = p_test.min(axis=1)
candidate_index = int(np.argmin(strict_probability))
source_row = test_frame.iloc[candidate_index]
before = Image.open(source_row.image_path).convert('RGB')
before_tensor = torch.from_numpy(
    np.asarray(before.resize((IMAGE_SIZE, IMAGE_SIZE)), dtype=np.float32) / 255.0
).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
working = before_tensor.clone().detach().requires_grad_(True)
pixel_optimizer = torch.optim.Adam([working], lr=0.01)
audit_trace = []
for iteration in range(31):
    logits = model(working)
    target = torch.ones_like(logits)
    success_loss = F.binary_cross_entropy_with_logits(logits, target)
    delta = working - before_tensor
    tv = (
        delta[:, :, 1:, :].sub(delta[:, :, :-1, :]).abs().mean()
        + delta[:, :, :, 1:].sub(delta[:, :, :, :-1]).abs().mean()
    )
    loss = success_loss + 0.10 * tv + 0.25 * delta.square().mean()
    audit_trace.append({
        'iteration': iteration, 'objective': float(loss.detach()),
        'predicted_min_probability': float(torch.sigmoid(logits).min().detach()),
        'delta_rms': float(delta.square().mean().sqrt().detach()),
    })
    if iteration == 30:
        break
    pixel_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    pixel_optimizer.step()
    with torch.no_grad():
        working.copy_(torch.max(torch.min(working, before_tensor + 8 / 255), before_tensor - 8 / 255))
        working.clamp_(0, 1)

after_array = (
    working.detach().cpu().squeeze(0).permute(1, 2, 0).numpy() * 255
).round().astype(np.uint8)
after = Image.fromarray(after_array).resize(before.size, Image.Resampling.LANCZOS)
before.save(RUN_DIR / 'audit-before.png')
after.save(RUN_DIR / 'audit-after.png')
(RUN_DIR / 'audit-trace.json').write_text(json.dumps(audit_trace, indent=2), encoding='utf-8')

expected_payload = str(source_row.expected_payload)
before_records = validator.validate(before, expected_payload)
after_records = validator.validate(after, expected_payload)
real_before = sum(item.exact_payload_match for item in before_records)
real_after = sum(item.exact_payload_match for item in after_records)
audit = {
    'real_before': real_before, 'real_after': real_after,
    'real_total': len(before_records),
    'surrogate_before': audit_trace[0]['predicted_min_probability'],
    'surrogate_after': audit_trace[-1]['predicted_min_probability'],
    'real_decoder_improved': real_after > real_before,
}
(RUN_DIR / 'gradient-audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
print(audit)
display(before.resize((420, 420)))
display(after.resize((420, 420)))


## 8. Porte de promotion, TorchScript et archive

In [ ]:
metric_gate = all(
    values['holdout_has_both_classes']
    and values['average_precision'] >= 0.75
    and values['brier'] <= 0.20
    for values in metrics.values()
)
physical_gate = physical_count > 0 or not REQUIRE_PHYSICAL_FOR_PRODUCTION
promotion = {
    'enough_data': True,
    'metric_gate': metric_gate,
    'real_decoder_gradient_gate': audit['real_decoder_improved'],
    'physical_gate': physical_gate,
    'physical_samples': physical_count,
}
promotion['research_usable'] = (
    promotion['metric_gate'] and promotion['real_decoder_gradient_gate']
)
promotion['production_usable'] = promotion['research_usable'] and promotion['physical_gate']

model = model.eval().cpu()
scripted = torch.jit.trace(model, torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE))
scripted.save(str(RUN_DIR / 'scan-surrogate.torchscript.pt'))
model_card = {
    'experiment': EXPERIMENT_NAME, 'decoder_outputs': decoder_names,
    'input': {'shape': [3, IMAGE_SIZE, IMAGE_SIZE], 'range': [0, 1]},
    'source_runs': [str(path) for path in SOURCE_RUNS],
    'split': 'GroupShuffleSplit by source_group; train/val/test disjoint',
    'metrics': metrics, 'promotion': promotion,
    'warning': 'Never replace external decoders or physical tests with this surrogate.',
}
(RUN_DIR / 'surrogate-card.json').write_text(json.dumps(model_card, indent=2), encoding='utf-8')
print('Promotion :', promotion)
if not promotion['production_usable']:
    print('NON PRODUCTION : conserver comme outil de recherche uniquement.')
archive = shutil.make_archive(str(RUN_DIR), 'gztar', RUN_DIR.parent, RUN_DIR.name)
print('Archive :', archive)
